<a href="https://colab.research.google.com/github/chlaitha/real-world-industry-projects/blob/main/Optimizing_Clinic_Attendance_with_Machine_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Optimizing Clinic Attendance with Machine Learning
**Introduction**: This project addresses healthcare operational inefficiencies by predicting patient no-shows using machine learning. By evaluating clinical records, demographic data, and scheduling lead times, the predictive engine identifies key risk factors behind missed appointments. This enables healthcare providers to implement targeted intervention strategies, optimize resource allocation, and improve overall patient attendance.

In [21]:
# Import libraries amd suppress warnings
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.inspection import permutation_importance

# Reproducibility and styling setup
SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [22]:
# Mount Google drive & load data
from google.colab import drive
drive.mount('/content/gdrive')

df = pd.read_csv('/content/gdrive/My Drive/Notebooks/healthcare.csv')

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).


In [23]:
df.head(10)

,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,2.987250e+13,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,5.589978e+14,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4.262962e+12,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,8.679512e+11,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8.841186e+12,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No
5,9.598513e+13,5626772,F,2016-04-27T08:36:51Z,2016-04-29T00:00:00Z,76,REPÚBLICA,0,1,0,0,0,0,No
6,7.336882e+14,5630279,F,2016-04-27T15:05:12Z,2016-04-29T00:00:00Z,23,GOIABEIRAS,0,0,0,0,0,0,Yes
7,3.449833e+12,5630575,F,2016-04-27T15:39:58Z,2016-04-29T00:00:00Z,39,GOIABEIRAS,0,0,0,0,0,0,Yes
8,5.639473e+13,5638447,F,2016-04-29T08:02:16Z,2016-04-29T00:00:00Z,21,ANDORINHAS,0,0,0,0,0,0,No
9,7.812456e+13,5629123,F,2016-04-27T12:48:25Z,2016-04-29T00:00:00Z,19,CONQUISTA,0,0,0,0,0,0,No


# 1. Cleaning and Preparing the Data
* **Data Type Conversion**: Convert ScheduledDay and AppointmentDay strings into standardized datetime objects. Ensure binary columns (Scholarship, Hipertension, Diabetes, Alcoholism, SMS_received) are properly formatted as integer indicators.

* **Timestamp Normalization**: Strip time components from ScheduledDay when computing lead time to match the date-only resolution of AppointmentDay.

* **Outlier & Anomaly Removal**: Filter out invalid age entries (such as negative values), clean typos in neighborhood names, and handle potential duplicate AppointmentID records.

* **Target Encoding**: Map No-show values ("Yes" / "No") into binary format (1 for missed appointments, 0 for attended).

In [24]:
# 1. Clean column names (fix typos)
df.rename(columns={
    'Hipertension': 'Hypertension',
    'Handcap': 'Handicap',
    'No-show': 'NoShow'
}, inplace=True)

# 2. Convert timestamps
df['ScheduledDay'] = pd.to_datetime(df['ScheduledDay'])
df['AppointmentDay'] = pd.to_datetime(df['AppointmentDay'])

# Extract date portion for accurate wait day calculation
df['ScheduledDate'] = df['ScheduledDay'].dt.date
df['AppointmentDate'] = df['AppointmentDay'].dt.date

# 3. Calculate waiting lead time (in days)
df['WaitDays'] = (df['AppointmentDate'] - df['ScheduledDate']).apply(lambda x: x.days)

# 4. Remove invalid entries (negative age, negative wait days)
df = df[(df['Age'] >= 0) & (df['Age'] <= 110)]
df = df[df['WaitDays'] >= 0]

# 5. Target encoding: 1 for No-Show, 0 for Attended
df['NoShow_Binary'] = df['NoShow'].map({'Yes': 1, 'No': 0})

print("Cleaned Dataset Shape:", df.shape)
df.head()

Cleaned Dataset Shape: (110516, 18)


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hypertension,Diabetes,Alcoholism,Handicap,SMS_received,NoShow,ScheduledDate,AppointmentDate,WaitDays,NoShow_Binary
0,2.987250e+13,5642903,F,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,JARDIM DA PENHA,0,1,0,0,0,0,No,2016-04-29,2016-04-29,0,0
1,5.589978e+14,5642503,M,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,0,0,0,0,0,No,2016-04-29,2016-04-29,0,0
2,4.262962e+12,5642549,F,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,MATA DA PRAIA,0,0,0,0,0,0,No,2016-04-29,2016-04-29,0,0
3,8.679512e+11,5642828,F,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No,2016-04-29,2016-04-29,0,0
4,8.841186e+12,5642494,F,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,JARDIM DA PENHA,0,1,1,0,0,0,No,2016-04-29,2016-04-29,0,0


# 2. Descriptive Statistics
* **Demographic Distributions**: Compute central tendency and spread (mean, median, standard deviation, interquartile range) for patient age across gender and neighborhood cohorts.

* **Prevalence Metrics**: Measure baseline rates for chronic conditions (Hypertension, Diabetes) and social assistance program (Scholarship) enrollment.

* **Operational Baselines**: Calculate the overall baseline appointment no-show rate and summary metrics for scheduling lead times (WaitDays).

In [25]:
# Baseline Attendance Rates
overall_noshow_rate = df['NoShow_Binary'].mean() * 100
print(f"Overall No-Show Rate: {overall_noshow_rate:.2f}%\n")

# Demographic Summary
demo_stats = df.groupby('Gender').agg(
    Total_Appointments=('AppointmentID', 'count'),
    Avg_Age=('Age', 'mean'),
    NoShow_Rate=('NoShow_Binary', 'mean')
).reset_index()
print("--- Gender Demographic Summary ---")
print(demo_stats, "\n")

# Health Profile & Assistance Breakdown
conditions = ['Hypertension', 'Diabetes', 'Alcoholism', 'Scholarship', 'SMS_received']
prevalence = df[conditions].mean().reset_index()
prevalence.columns = ['Condition/Indicator', 'Prevalence_Rate']
print("--- Chronic Conditions & Program Prevalence ---")
print(prevalence, "\n")

# Lead Time vs No-Show Stats
wait_stats = df.groupby('NoShow')[['WaitDays', 'Age']].describe()
print("--- Summary Statistics by Attendance Status ---")
print(wait_stats)

Overall No-Show Rate: 20.19%

--- Gender Demographic Summary ---
  Gender  Total_Appointments    Avg_Age  NoShow_Rate
0      F               71831  38.889170     0.203088
1      M               38685  33.737443     0.199638 

--- Chronic Conditions & Program Prevalence ---
  Condition/Indicator  Prevalence_Rate
0        Hypertension         0.197257
1            Diabetes         0.071872
2          Alcoholism         0.030403
3         Scholarship         0.098275
4        SMS_received         0.321049 

--- Summary Statistics by Attendance Status ---
       WaitDays                                                         Age  \
          count       mean        std  min  25%   50%   75%    max    count   
NoShow                                                                        
No      88205.0   8.754787  14.550570  0.0  0.0   2.0  12.0  179.0  88205.0   
Yes     22311.0  15.835642  16.605608  0.0  4.0  11.0  23.0  179.0  22311.0   

                                              

# 3. Predictive Modeling: No-Show Risk Classification

* **Business Problem**: Unscheduled clinic absenteeism leads to underutilized medical staff, lost facility revenue, and delayed care for other patients. Predicting which patients are likely to miss appointments allows clinics to proactively send reminders or optimize booking schedules.

* **Analysis Summary**:
  * **Feature Engineering**: Calculate WaitDays (difference between scheduling date and appointment date), extract appointment day-of-week, and build an aggregated chronic illness index combining hypertension, diabetes, and alcoholism indicators.

  * **Model Implementation**: Train classification models (such as Logistic Regression, Random Forests, and LightGBM) to output individual patient no-show probabilities.

  * **Performance Evaluation**: Assess model accuracy using ROC-AUC, Precision-Recall curves, and F1-scores to account for class imbalance in appointment absenteeism.

In [26]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score

# Feature Engineering
df['DayOfWeek'] = df['AppointmentDay'].dt.dayofweek
df['ComorbidityIndex'] = df['Hypertension'] + df['Diabetes'] + df['Alcoholism']

features = ['Age', 'WaitDays', 'Scholarship', 'Hypertension',
            'Diabetes', 'Alcoholism', 'Handicap', 'SMS_received',
            'DayOfWeek', 'ComorbidityIndex']

X = df[features]
y = df['NoShow_Binary']

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Model Training
model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# Predictions & Evaluation
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("ROC-AUC Score:", roc_auc_score(y_test, y_proba))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

ROC-AUC Score: 0.6360425167092647

Classification Report:
               precision    recall  f1-score   support

           0       0.83      0.80      0.81     17642
           1       0.32      0.37      0.34      4462

    accuracy                           0.71     22104
   macro avg       0.58      0.59      0.58     22104
weighted avg       0.73      0.71      0.72     22104



# 4. Causal Inference & Intervention Analysis

* **Business Problem**: Clinic managers need to verify whether specific intervention programs—such as SMS notifications or welfare support—causally improve patient attendance rather than merely correlating with higher-income or younger demographics

* **Analysis Summary**:
  * **Confounder Control**: Apply Propensity Score Matching (PSM) or Double Machine Learning (DML) to control for confounding variables like patient age, neighborhood background, and baseline health conditions.

  * **Treatment Effect Estimation**: Quantify the Average Treatment Effect (ATE) of receiving an SMS reminder (SMS_received) on reducing no-shows.

  * **Program Evaluation**: Measure whether enrollment in the Scholarship welfare program independently impacts attendance rates after balancing underlying demographic disparities.

In [27]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Propensity Score Estimation using Logistic Regression
ps_model = smf.logit(
    'SMS_received ~ Age + WaitDays + Scholarship + Hypertension + Diabetes',
    data=df
).fit()

df['propensity_score'] = ps_model.predict(df)

# OLS Regression with Propensity Score Control (Doubly Robust Approach)
causal_model = smf.ols(
    'NoShow_Binary ~ SMS_received + propensity_score + Age + WaitDays',
    data=df
).fit(cov_type='HC3')

print(causal_model.summary())
# The coefficient for SMS_received represents the estimated causal effect controlling for key confounders.

Optimization terminated successfully.
         Current function value: 0.544707
         Iterations 6
                            OLS Regression Results                            
Dep. Variable:          NoShow_Binary   R-squared:                       0.047
Model:                            OLS   Adj. R-squared:                  0.047
Method:                 Least Squares   F-statistic:                     1146.
Date:                Mon, 07 Sep 2026   Prob (F-statistic):               0.00
Time:                        13:47:19   Log-Likelihood:                -53296.
No. Observations:              110516   AIC:                         1.066e+05
Df Residuals:                  110511   BIC:                         1.067e+05
Df Model:                           4                                         
Covariance Type:                  HC3                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
-----------------------

# 5. Operational Optimization & Capacity Planning

* **Business Problem**: Fixed daily appointment schedules leave clinics vulnerable to lost capacity during high no-show days, while unmanaged overbooking risks overworking staff and increasing patient wait times.

* **Analysis Summary**:
  * **Risk Stratification**: Categorize scheduled appointments into low, medium, and high no-show risk tiers using model outputs.

  * **Dynamic Overbooking Strategy**: Design data-driven overbooking thresholds based on predicted no-show volume per neighborhood and time slot.

  * **Resource Allocation**: Prioritize high-cost direct interventions (such as automated voice calls or community health worker check-ins) specifically for top-tier high-risk patients.

In [28]:
# Predict risk probabilities across full dataset
df['Predicted_NoShow_Prob'] = model.predict_proba(X)[:, 1]

# Risk Stratification into Action Tiers
def assign_risk_tier(prob):
    if prob >= 0.7:
        return 'High Risk'
    elif prob >= 0.4:
        return 'Medium Risk'
    else:
        return 'Low Risk'

df['Risk_Tier'] = df['Predicted_NoShow_Prob'].apply(assign_risk_tier)

# Capacity Planning & Intervention Summary
capacity_summary = df.groupby('Risk_Tier').agg(
    Total_Patients=('AppointmentID', 'count'),
    Avg_Predicted_Prob=('Predicted_NoShow_Prob', 'mean'),
    Actual_NoShow_Rate=('NoShow_Binary', 'mean')
).reindex(['Low Risk', 'Medium Risk', 'High Risk'])

print("--- Operational Risk Tiers for Staffing & Intervention ---")
print(capacity_summary)

--- Operational Risk Tiers for Staffing & Intervention ---
             Total_Patients  Avg_Predicted_Prob  Actual_NoShow_Rate
Risk_Tier                                                          
Low Risk              73148            0.112752            0.054684
Medium Risk           24645            0.549308            0.365389
High Risk             12723            0.809476            0.731431


# 6. Key Findings & Executive Summary

Approximately 1 out of every 5 scheduled medical appointments ends in a no-show (overall rate: 20.19%). This absenteeism severely limits patient access, creates idle clinic capacity, and inflates operational costs.

### Primary Insights:

  * **Lead Time is the Main Driver**: The duration between booking an appointment and the actual visit is the strongest predictor of attendance. Patients who show up wait an average of 8.8 days, whereas patients who miss their appointments wait nearly twice as long (15.8 days). Same-day and short-notice bookings have near-zero no-show rates.

  * **Demographic Parity**: Men (19.96%) and women (20.31%) miss appointments at almost identical rates, though women account for roughly 65% of total appointment volume. Slightly younger patients exhibit a marginally higher likelihood of missing appointments.

  * **SMS paradox**: Receiving an text reminder correlates with a higher likelihood of missing an appointment (+4.5 percentage points). This occurs because reminders are predominantly dispatched for longer lead-time appointments, which already carry a naturally high risk of absenteeism.

  * **High-Risk Concentration**: Machine learning models successfully isolate high-risk visits. While low-risk patients miss only 5.5% of appointments, patients placed in the High Risk tier miss their visits 73.1% of the time.

# 7. Actionable Strategies & Implementation Plan

To reduce appointment leakage and improve facility utilization, clinics can transition from passive scheduling to targeted, risk-based operational workflows.

### 1. Implement Tiered Smart-Scheduling & Dynamic Overbooking
  * **Strategy**: Use the high-risk classification model to dynamically overbook high-risk slots while leaving low-risk slots strictly one-to-one.

  * **Action Plan**:

    * Low Risk (<20% predicted probability): Book single appointments standardly.

    * High Risk (>70% predicted probability): Double-book these time slots or pair them with walk-in buffers. Since 73% of this group misses their visit, overbooking balances out operational idle time without overloading doctors.

### 2. Restructure Lead-Time Workflows

  * **Strategy**: Cap forward-booking lead times to mitigate the steep rise in absenteeism past 7 days.

  * **Action Plan**:

    * For appointments booked more than two weeks out, require a mandatorily confirmed double check-in 3 days prior to keep the slot active.

    * High Risk (>70% predicted probability): Double-book these time slots or pair them with walk-in buffers. Since 73% of this group misses their visit, overbooking balances out operational idle time without overloading doctors.

### 3. Redesign SMS & Outreach Interventions

  * **Strategy**: Move from plain broadcast notifications to interactive, multi-channel reminders for long-wait patients.

  * **Action Plan**:

    * Replace static SMS blasts with 2-way text messaging (e.g., "Reply 1 to Confirm, 2 to Reschedule"). Automatically open cancelled slots for standby patients.

    * Assign high-risk, chronic-care patients (hypertension/diabetes) to personal phone follow-ups or community health worker check-ins rather than automated text messages.
